In [2]:
import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from hyperopt import fmin, tpe, Trials, STATUS_OK, space_eval

In [4]:
data = pd.read_csv("../datasets/data.csv", parse_dates=["timestamp"], index_col="timestamp")
data.head()

,production_total,consommation_totale,temp
timestamp,,,
2022-12-31 23:00:00,40343.0,45622.00,15.0
2023-01-01 00:00:00,37822.0,45611.25,15.0
2023-01-01 01:00:00,36931.0,44854.00,14.8
2023-01-01 02:00:00,36016.0,41910.00,14.6
2023-01-01 03:00:00,35569.0,39449.25,14.3


In [5]:
data_prod = data["production_total"].copy()

In [23]:
horizon = 1
lookback = 24
X = []
y = []


for i in range(lookback, len(data_prod) - horizon ):
    X.append(data_prod.iloc[i - lookback:i].values)
    y.append(data_prod.iloc[i:i + horizon].values)

X = np.array(X)
y = np.array(y)

In [24]:
X.shape, y.shape

((29902, 24), (29902, 1))

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=42)

In [26]:
model = LGBMRegressor()
model.fit(X_train, y_train)

d:\Nouveau dossier\TitreRNCP_Bloc1\projet_prevision_energies_bloc5\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.040957 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6120
[LightGBM] [Info] Number of data points in the train set: 23921, number of used features: 24
[LightGBM] [Info] Start training from score 58348.900234


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [27]:
y_pred = model.predict(X_test)

In [28]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae, mse

(1614.798993659903, 18021720.743511416)

In [ ]:
from skopt import space


def evaluate_models(X_train, y_train,
                    X_valid, y_valid,
                    models, param, max_evals: int = 15):
    """Optimisation bayésienne des hyperparamètres (hyperopt / TPE).

    Pour chaque modèle, minimise la MAE de validation via l'algorithme TPE
    (Tree-structured Parzen Estimator) au lieu d'un balayage exhaustif
    (GridSearchCV). Chaque modèle de `models` est réajusté en place avec ses
    meilleurs hyperparamètres, et la fonction renvoie {nom_modele: MAE_valid}.

    param : {nom_modele: espace de recherche hyperopt (hp.*)}.
    """
    try:
        # Import paresseux : hyperopt n'est requis que pour l'entraînement.
        from hyperopt import fmin, tpe, Trials, STATUS_OK, space_eval

        report = {}

        model = LGBMRegressor()

        def objective(candidate, _model=model):
                # MAE de validation pour un jeu d'hyperparamètres candidat.
                _model.set_params(**candidate)
                _model.fit(X_train, y_train)
                pred = _model.predict(X_valid)
                return {"loss": mean_absolute_error(y_valid, pred), "status": STATUS_OK}

        trials = Trials()
        best = fmin(
                fn=objective,
                space=param,
                algo=tpe.suggest,
                max_evals=max_evals,
                trials=trials,
                show_progressbar=False,
                rstate=np.random.default_rng(42),
            )
        best_params = space_eval(space, best)

            # Réajustement final avec les meilleurs hyperparamètres.
        model.set_params(**best_params)
        model.fit(X_train, y_train)
        test_model_score = mean_absolute_error(y_valid, model.predict(X_valid))
            
        return test_model_score, best_params

    except ImportError:
        print("hyperopt n'est pas installé. Installez-le pour l'optimisation bayésienne.")
        return {}


In [31]:
test_model_score, best_params = evaluate_models(X_train, y_train, X_test, y_test, models=None, param=None)

TypeError: fmin() missing 1 required positional argument: 'space'